# GPUs for Deep Learning

Companion notebook for the [GPUs for Deep Learning lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/04-gpus-for-deep-learning).

**The idea in one sentence.** Deep learning is fast on GPUs because its core
operation — **matmul** — has high arithmetic intensity (lots of FLOPs per byte), and
four levers push it further into the compute-bound regime where GPUs shine:
**bigger matmuls**, **mixed precision**, **kernel fusion**, and **batching**.

The four levers, quantified from scratch:

- **Matmul intensity grows with size** — big matmuls are compute-bound.
- **Mixed precision (FP16)** halves the bytes → doubles intensity and memory headroom.
- **Kernel fusion** collapses memory round-trips without changing the math.
- **Batching** rescues memory-bound decode by reusing the loaded weights.

We **validate each lever numerically** (and that fusion preserves the result), then
cover the gotchas — including mixed-precision loss scaling.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Matmul intensity grows with size

A square `N x N` matmul costs `2N^3` FLOPs and moves `~3 * N^2 * bytes_per_elem` bytes (two inputs +
one output). So arithmetic intensity `I = FLOPs / bytes` grows linearly with `N` — large matmuls
drift firmly into compute-bound territory, the GPU's strong suit.

In [ ]:
def matmul_flops(M, N, K):
    return 2 * M * N * K

def matmul_intensity(N, bytes_per_elem=4):
    flops = matmul_flops(N, N, N)
    bytes_moved = 3 * N * N * bytes_per_elem        # A, B, C
    return flops / bytes_moved

sizes = np.array([16, 32, 64, 128, 256, 512, 1024, 2048, 4096])
I = [matmul_intensity(n) for n in sizes]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.loglog(sizes, I, 'o-', color='#2dd4bf')
ax.axhline(20, ls='--', color='#fb7185', label='example ridge point (20 FLOP/byte)')
ax.set_xlabel('matrix size N'); ax.set_ylabel('arithmetic intensity (FLOP/byte)')
ax.set_title('Bigger matmuls are more compute-bound (intensity grows with N)')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"N=64   intensity = {matmul_intensity(64):.1f} FLOP/byte")
print(f"N=4096 intensity = {matmul_intensity(4096):.0f} FLOP/byte  (deeply compute-bound)")

### Validate: matmul intensity grows with size

An $N\times N$ matmul does $2N^3$ FLOPs but moves only $\sim 3N^2$ elements, so
arithmetic intensity scales like $N$ — bigger matmuls are more compute-bound. We
confirm intensity increases monotonically with $N$ and crosses an example ridge.

In [ ]:
I_vals = [matmul_intensity(n) for n in sizes]
print('N -> intensity (FLOP/byte):')
for n, iv in zip(sizes, I_vals):
    print(f'  N={n:5d}: {iv:8.1f}  [{"compute" if iv > 20 else "memory"}-bound at ridge 20]')
assert all(I_vals[i] < I_vals[i+1] for i in range(len(I_vals)-1)), 'intensity must grow with N'
assert I_vals[0] < 20 < I_vals[-1], 'small matmuls are memory-bound, large ones compute-bound'
print('\n✅ intensity ~ N/6, so scaling up the matmul moves it into the compute-bound regime')

## 2 — Mixed precision halves the bytes

Doing matmuls in 16-bit (FP16/BF16) instead of FP32 halves the bytes moved for the same FLOPs,
**doubling** arithmetic intensity for a bandwidth-limited op — and engaging tensor cores on top.

Saving activation memory isn't limited to lower precision — the companion notebook **[`notebooks/wiki/gradient-checkpointing.ipynb`](https://ml-viz-ruby.vercel.app/wiki/gradient-checkpointing)** trades recompute for memory in a complementary, orthogonal way.

In [ ]:
N = 1024
fp32 = matmul_intensity(N, bytes_per_elem=4)
fp16 = matmul_intensity(N, bytes_per_elem=2)
print(f"FP32 intensity: {fp32:.0f} FLOP/byte")
print(f"FP16 intensity: {fp16:.0f} FLOP/byte  ({fp16/fp32:.1f}x higher -> better for memory-bound regimes)")

# activations/gradients memory for a hidden state
elems = 8 * 2048 * 4096        # batch x seq x hidden
print(f"\nactivation tensor: {elems/1e6:.0f}M elements")
print(f"  FP32: {elems*4/1e9:.2f} GB")
print(f"  FP16: {elems*2/1e9:.2f} GB  (half the memory and bandwidth)")

### Validate: FP16 exactly doubles arithmetic intensity

Halving bytes-per-element (FP32→FP16) leaves the FLOPs unchanged but halves bytes
moved, so intensity exactly doubles — and every activation/gradient tensor takes half
the memory and bandwidth. That's why mixed precision is free speed in memory-bound
regimes.

In [ ]:
for N in [512, 1024, 2048]:
    r = matmul_intensity(N, 2) / matmul_intensity(N, 4)
    print(f'N={N}: FP16/FP32 intensity ratio = {r:.2f}')
    assert np.isclose(r, 2.0), 'FP16 should exactly double intensity'
print('\n✅ mixed precision doubles intensity and halves memory — the default for training')

## 3 — Kernel fusion collapses memory round-trips

A chain of `k` element-wise ops each reads its input from global memory and writes its output back:
`2k` round-trips. Fusing them does one load, all the arithmetic in registers, one store: `2` trips.
For these memory-bound chains, that approaches a `k`x speedup.

In [ ]:
def roundtrips_unfused(k):
    return 2 * k

def roundtrips_fused(k):
    return 2

for k in [1, 2, 3, 5]:
    u, f = roundtrips_unfused(k), roundtrips_fused(k)
    print(f"chain of {k} ops:  unfused={u} trips  fused={f} trips  -> {u/f:.1f}x less memory traffic")

# numeric demo: bias -> relu -> scale, fused vs staged, same result
x = np.random.default_rng(0).normal(size=10000).astype(np.float32)
bias, scale = 0.5, 2.0
staged = x + bias; staged = np.maximum(staged, 0); staged = staged * scale
fused = np.maximum(x + bias, 0) * scale          # one pass, no intermediates stored
assert np.allclose(staged, fused)
print("\n\u2713 fused result matches staged result (fusion changes traffic, not math)")

**What to notice — fusion changes traffic, not math.** The fused `relu(x+bias)*scale`
gives the *identical* result to the staged version (asserted above), but a chain of
$k$ ops does 2 memory round-trips instead of $2k$ — the win grows linearly with the
chain length.

## 4 — Batching rescues memory-bound decode

Generating one token re-reads the whole weight matrix to produce a single output column — tiny
intensity. Batching `B` requests reuses each loaded weight across `B` tokens, so intensity scales
with the batch size, pushing decode from memory-bound toward compute-bound.

In [ ]:
def decode_intensity(batch, d_model=4096, bytes_per_elem=2):
    # weight matrix d_model x d_model applied to a batch of vectors
    flops = 2 * batch * d_model * d_model
    bytes_moved = (d_model * d_model + 2 * batch * d_model) * bytes_per_elem  # weights + in/out
    return flops / bytes_moved

batches = [1, 2, 4, 8, 16, 32, 64, 128, 256]
Ib = [decode_intensity(b) for b in batches]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogx(batches, Ib, 'o-', color='#818cf8', base=2)
ax.axhline(20, ls='--', color='#fb7185', label='example ridge (20 FLOP/byte)')
ax.set_xlabel('batch size'); ax.set_ylabel('arithmetic intensity (FLOP/byte)')
ax.set_title('Batching raises decode intensity: memory-bound -> compute-bound')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"batch 1:   intensity = {decode_intensity(1):.2f} FLOP/byte (memory-bound, GPU idle)")
print(f"batch 256: intensity = {decode_intensity(256):.0f} FLOP/byte (compute-bound, GPU busy)")

### Validate: batching lifts memory-bound decode above the ridge

LLM decode with batch size 1 re-reads the whole weight matrix to process a single
token — hopelessly memory-bound. Increasing the batch amortises that weight load over
many tokens, raising intensity until it crosses the compute ridge. We confirm
intensity rises with batch and eventually becomes compute-bound.

In [ ]:
ints = [decode_intensity(b) for b in batches]
print('batch -> decode intensity (FLOP/byte):')
for b, iv in zip(batches, ints):
    print(f'  batch {b:3d}: {iv:6.1f}  [{"compute" if iv > 20 else "memory"}-bound]')
assert ints[0] < ints[-1], 'batching should raise arithmetic intensity'
assert ints[0] < 20 < ints[-1], 'batch=1 is memory-bound; a large batch is compute-bound'
print('\n✅ batching amortises the weight load over many tokens — why servers batch decode')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **FP16 underflow** | small gradients vanish → use loss scaling (demo) or BF16's wider range |
| **fusion limits** | only elementwise/adjacent ops fuse cheaply; a matmul in the middle breaks the chain |
| **batching adds latency** | larger batches raise throughput but each request waits longer |
| **tensor cores need aligned shapes** | dimensions off the tensor-core tile size leave performance on the table |
| **accumulation precision** | reductions in FP16 lose accuracy; accumulate in FP32 |

Demo: FP16 gradient underflow and the loss-scaling fix.

In [ ]:
# Mixed precision has a catch: FP16 has a tiny dynamic range, so small gradients
# UNDERFLOW to zero. Loss scaling multiplies the loss by a large factor before backprop
# (shifting gradients into FP16's representable range), then unscales — the trick behind
# every mixed-precision trainer.
fp16_min_normal = np.float16(6.1e-5)                    # smallest normal FP16
tiny_grad = 1e-8                                        # a realistic small gradient
print(f'raw gradient {tiny_grad:.1e} as FP16: {np.float16(tiny_grad)}  (underflows to 0!)')
scaled = np.float16(tiny_grad * 1024)                  # scale the loss by 1024
print(f'gradient * 1024 as FP16: {scaled}  (now representable)')
print(f'unscaled back to FP32: {float(scaled) / 1024:.2e}  (recovered)')
assert np.float16(tiny_grad) == 0 and scaled != 0
print('\nLoss scaling is what makes FP16 training numerically safe — see the DML exercise below.')

## ✏️ Your turn

**Exercise.** Implement `fusion_speedup(k)` — the factor by which fusing a chain of `k` memory-bound
element-wise ops reduces global-memory traffic (round-trips unfused ÷ fused) — and
`bytes_saved(k, tensor_bytes)`, the global-memory bytes saved per pass given each op's tensor is
`tensor_bytes` (unfused moves `2k * tensor_bytes`; fused moves `2 * tensor_bytes`).

In [ ]:
def fusion_speedup(k):
    # TODO(you): unfused round-trips / fused round-trips
    return ...

def bytes_saved(k, tensor_bytes):
    # TODO(you): (unfused trips - fused trips) * tensor_bytes
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert fusion_speedup(1) == 1.0           # nothing to fuse
assert fusion_speedup(5) == 5.0           # 10 trips -> 2 trips
assert bytes_saved(1, 1_000_000) == 0     # single op saves nothing
assert bytes_saved(3, 1_000_000) == 4_000_000   # (6 - 2) * 1MB
assert matmul_flops(128, 512, 256) == 33_554_432
print("\u2713 fusion + FLOP accounting checks pass")

<details>
<summary>Solution</summary>

```python
def fusion_speedup(k):
    return (2 * k) / 2          # = k

def bytes_saved(k, tensor_bytes):
    return (2 * k - 2) * tensor_bytes
```

Fusion turns `2k` global round-trips into `2`, a `k`x reduction for memory-bound chains. This is
exactly what `torch.compile`, XLA, and FlashAttention do — and why the giant attention score matrix
never needs to touch global memory.

</details>

---
## 🔬 Extra practice — DML `160_mixed_precision_training`

[Open-Deep-ML problem 160](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/160_mixed_precision_training) asks for a small `MixedPrecision` class that bundles the two bookkeeping pieces real mixed-precision training needs: a **forward pass computed in FP16** (cast weights/inputs/targets down, matmul + MSE loss in FP16) whose loss is then **scaled up** before backprop, and a **backward pass** that unscales incoming gradients back to FP32 and zeroes them out if scaling pushed anything to `NaN`/`Inf` (an overflow the optimizer step must skip). Match its exact signature:

```python
class MixedPrecision:
    def __init__(self, loss_scale=1024.0): ...
    def forward(self, weights, inputs, targets): ...   # -> scaled loss, as a plain float
    def backward(self, gradients): ...                  # -> unscaled float32 gradients
```

`forward` always casts to `float16` for the compute (regardless of the input arrays' own dtype) and returns the **scaled** loss as a plain Python `float`. `backward` always casts incoming gradients to `float32`, and returns an all-zero `float32` array if it detects overflow instead of unscaled garbage.

In [ ]:
class MixedPrecision:
    """DML 160: mixed-precision training bookkeeping (FP16 forward + loss
    scaling + FP32 gradient unscaling with overflow detection)."""

    def __init__(self, loss_scale=1024.0):
        # TODO(you): store the loss-scale factor
        self.loss_scale = ...

    def forward(self, weights, inputs, targets):
        # TODO(you): cast weights/inputs/targets to float16, run the linear
        # forward pass (inputs @ weights) + MSE loss in float16, then scale
        # the loss by self.loss_scale and return it as a plain Python float
        weights_fp16 = ...
        inputs_fp16 = ...
        targets_fp16 = ...
        predictions = ...
        loss = ...
        return ...

    def backward(self, gradients):
        # TODO(you): cast gradients to float32, detect overflow (any NaN or
        # Inf) -- if overflow, return zeros (float32, same shape); otherwise
        # unscale by self.loss_scale and return as float32
        gradients_fp32 = ...
        overflow = ...
        if overflow:
            return ...
        return ...

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json, plus two
# edge cases: FP16 underflow without scaling, and loss_scale=1 as a no-op)
mp = MixedPrecision(loss_scale=1024.0)
weights = np.array([0.5, -0.3], dtype=np.float32)
inputs = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
targets = np.array([1.0, 0.0], dtype=np.float32)
loss = mp.forward(weights, inputs, targets)
assert f"{loss:.4f}" == "665.0000"
assert isinstance(loss, float)

grads = np.array([512.0, -256.0], dtype=np.float32)
result = mp.backward(grads)
assert np.allclose(result, [0.5, -0.25])
assert result.dtype == np.float32

# a second (loss_scale, dtype) combination, straight from DML's test bank
mp2 = MixedPrecision(loss_scale=512.0)
w64 = np.array([1.0, 0.5], dtype=np.float64)
i64 = np.array([[2.0, 1.0]], dtype=np.float64)
t64 = np.array([3.0], dtype=np.float64)
assert f"{mp2.forward(w64, i64, t64):.1f}" == "128.0"
result2 = mp2.backward(np.array([1024.0, 512.0], dtype=np.float16))
assert np.allclose(result2, [2.0, 1.0]) and result2.dtype == np.float32

# overflow handling: NaN and Inf gradients must both zero out, whatever dtype they arrive as
assert np.allclose(MixedPrecision(2048.0).backward(np.array([np.nan], dtype=np.float16)), [0.0])
assert np.allclose(MixedPrecision(256.0).backward(np.array([np.inf], dtype=np.float64)), [0.0])

# Edge case -- loss_scale=1.0 is a no-op: forward matches a plain (unscaled)
# FP16 computation exactly, and backward returns gradients unchanged.
mp_noop = MixedPrecision(loss_scale=1.0)
unscaled_loss = mp_noop.forward(weights, inputs, targets)
raw_fp16_loss = float(np.mean((targets.astype(np.float16) -
                                inputs.astype(np.float16) @ weights.astype(np.float16)) ** 2))
assert abs(unscaled_loss - raw_fp16_loss) < 1e-3
g = np.array([0.01, -0.02], dtype=np.float32)
assert np.allclose(mp_noop.backward(g), g)

# Edge case -- *why* loss scaling exists: a small real gradient underflows to
# zero if cast straight to FP16, but scaling it up first (as forward()/the
# optimizer would, via the scaled loss) lifts it into FP16's representable
# range, and backward() recovers the true tiny value on the way out.
tiny_grad = np.array([2e-8], dtype=np.float32)
assert tiny_grad.astype(np.float16)[0] == 0.0, "unscaled: underflows to zero in FP16"
mp_scaled = MixedPrecision(loss_scale=1024.0)
scaled_grad_fp16 = (tiny_grad * mp_scaled.loss_scale).astype(np.float16)
assert scaled_grad_fp16[0] != 0.0, "scaled: representable in FP16"
recovered = mp_scaled.backward(scaled_grad_fp16.astype(np.float32))
assert np.allclose(recovered, tiny_grad, atol=1e-9), "backward() unscales back to the true gradient"

print("✅ DML 160 mixed_precision_training passed (incl. underflow + loss_scale=1 no-op checks)")

<details>
<summary>💡 Show solution</summary>

```python
class MixedPrecision:
    def __init__(self, loss_scale=1024.0):
        self.loss_scale = loss_scale

    def forward(self, weights, inputs, targets):
        weights_fp16 = weights.astype(np.float16)
        inputs_fp16 = inputs.astype(np.float16)
        targets_fp16 = targets.astype(np.float16)
        predictions = np.dot(inputs_fp16, weights_fp16)
        loss = np.mean((targets_fp16 - predictions) ** 2)
        return float(loss) * self.loss_scale

    def backward(self, gradients):
        gradients_fp32 = gradients.astype(np.float32)
        overflow = np.any(np.isnan(gradients_fp32)) or np.any(np.isinf(gradients_fp32))
        if overflow:
            return np.zeros_like(gradients_fp32, dtype=np.float32)
        return (gradients_fp32 / self.loss_scale).astype(np.float32)
```

`forward` always computes in FP16 regardless of the caller's input dtype (that's the whole
point — a real training loop keeps FP32 "master weights" but runs the matmul in FP16 for
speed/memory), then scales the loss up before it's handed to `.backward()` on a real
autodiff graph, so tiny gradients don't get flushed to zero before they're unscaled again
here. `backward` unscales, but only after checking `isnan`/`isinf` — if the scaled loss
already overflowed on the way *up*, unscaling `NaN`/`Inf` would still be `NaN`/`Inf`, so the
optimizer step must be skipped by zeroing the gradients instead of applying garbage.

</details>

## Key takeaways

- **Matmul is GPU-friendly** because its intensity grows with size (verified) —
  bigger matmuls are compute-bound.
- **Mixed precision (FP16) doubles intensity and halves memory** (verified) — free
  speed, guarded by **loss scaling** against underflow (demo).
- **Kernel fusion** collapses $2k$ memory round-trips to $2$ without changing the math
  (verified identical result).
- **Batching rescues memory-bound decode** by amortising the weight load over many
  tokens, pushing it past the compute ridge (verified) — why inference servers batch.